# K2DAREK ablation study (extend=True variant): MLP block, spectral norm, spline block, knot selection, and Lipschitz sharing on kdk itself

Variant of `Fig2_cos_k2dk_journal_ablation_arch.ipynb` with two changes:

1. **`kan_extend=True`/`extend=True` everywhere** (the KAN spline grid's boundary-knot
   padding is on for every arm, matching the main notebook's "Ext" variants), instead of the
   original notebook's `extend=False` baseline.
2. **The Lipschitz-budget / error-share allocation axis now also runs on `kdk` (full K2DAREK)
   itself**, not just the standalone no-MLP `DAREK`. Since the paper's claims are about K2DAREK
   specifically, this directly tests whether the allocation-strategy story holds for the actual
   architecture, not just its spline sub-component in isolation. A `Random_Lipschitz` baseline
   (new, added to `Lipschitz_share.py`) was also added alongside Equal/Heuristic/DataDriven/
   Worst-case/SHAP, to have a naive "non-uniform but uninformed" contrast against the smarter
   allocation strategies.

Everything else (architecture, benchmark, metrics, evaluation harness) is identical to the
original ablation notebook -- see that notebook's first cell for the full axis rundown.

In [ ]:
### Load data
import numpy as np
import matplotlib.pyplot as plt
import torch
import random

from kdarek import Dataset, DAREK, KDAREK as K2DAREK
from kdarek.lipschitz_share import (Equal_Lipschitz, Heuristic_Lipschitz,
                              DataDriven_Lipschitz, NonOptimal_WorstCase_Lipschitz)
from kdarek.error_share import SHAP_Error_Share, Apprx_SHAP_Error_Share
from tabulate import tabulate

seed = 12
cos_dataset = Dataset(fx=lambda x: 10 * np.cos(x), n=50, fix=True, seed=seed)
plt.scatter(cos_dataset['train_input'], cos_dataset['train_label'], label='train', alpha=0.5)
plt.scatter(cos_dataset['test_input'], cos_dataset['test_label'], label='test', alpha=0.05)
plt.legend()


## Shared evaluation harness

Reused/extended from `Fig2_cos_k2dk_journal_std.ipynb` cell 2.

In [ ]:
def set_seed(s):
    torch.manual_seed(s)
    np.random.seed(s)
    random.seed(s)

x_train, y_train = cos_dataset['train_input'], cos_dataset['train_label']
x_test, y_test   = cos_dataset['test_input'], cos_dataset['test_label']

# dense, sorted grid over the test domain, used for the finite-difference metrics below
xindx   = x_test.sort(dim=0)[1][:, 0]
x_dense = x_test[xindx]

def check_error_violation(xt, yt, yhat, uhat, eps=1e-5):
    lb = yhat - uhat
    ub = yhat + uhat
    vio   = 1 - np.bitwise_and(lb < (yt + eps), yt < (ub + eps)).sum() / yt.shape[0]
    error = np.sqrt(((yhat - yt) ** 2).mean())
    return error, vio, uhat.mean()

def empirical_lipschitz(x_sorted, y_sorted):
    """max |dy/dx| via finite differences -- checks the fitted curve against its prescribed
    Lipschitz budget (the paper's 'better Lipschitz control' claim)."""
    x_sorted, y_sorted = x_sorted.flatten(), y_sorted.flatten()
    dx = x_sorted[1:] - x_sorted[:-1]
    dy = y_sorted[1:] - y_sorted[:-1]
    return (dy.abs() / dx.abs().clamp_min(1e-8)).max()

def curvature_score(x_sorted, y_sorted):
    """mean |second finite difference| -- directly targets the 'smoother approximation'
    claim: lower is smoother."""
    x_sorted, y_sorted = x_sorted.flatten(), y_sorted.flatten()
    dx  = (x_sorted[2:] - x_sorted[:-2]) / 2
    d2y = y_sorted[2:] - 2 * y_sorted[1:-1] + y_sorted[:-2]
    return (d2y.abs() / dx.clamp_min(1e-8) ** 2).mean()

N = 10
set_seed(0)
base_seeds = np.random.random_integers(0, 1e6, N)  # trial i uses base_seeds[i], reproducible per trial

results = {}

def new_arm(key):
    results[key] = {'er': [], 'vio': [], 'umean': [], 'lip': [], 'smooth': []}

def record_arm(key, yhat, uhat):
    er, vio, umean = check_error_violation(x_test, y_test, yhat, uhat)
    yhat_dense = yhat[xindx]
    lip    = empirical_lipschitz(x_dense, yhat_dense)
    smooth = curvature_score(x_dense, yhat_dense)
    results[key]['er'].append(float(er));     results[key]['vio'].append(float(vio))
    results[key]['umean'].append(float(umean))
    results[key]['lip'].append(float(lip));   results[key]['smooth'].append(float(smooth))

def print_ablation_table():
    order = [
        ('K2DAREK (full)',                   'kdk_full'),
        ('No MLP block (spline only)',       'dk_nomlp'),
        ('No spectral norm',                 'kdk_nospec'),
        ('Knots: random',                    'kdk_knot_random'),
        ('Knots: LHS',                       'kdk_knot_lhs'),
        ('Knots: CHB',                       'kdk_knot_chb'),
        ('Knots: WKM',                       'kdk_knot_wkm'),
        ('Knots: fixed (frozen)',            'kdk_knot_fixed'),
        ('Spline k=1',                       'kdk_k1'),
        ('Spline knots=5 (small)',            'kdk_grid_small'),
        ('Spline knots=15 (large)',           'kdk_grid_large'),
        ('No-MLP + Equal Lipschitz',         'dk_lip_equal'),
        ('No-MLP + Random Lipschitz',        'dk_lip_random'),
        ('No-MLP + Worst-case Lipschitz',    'dk_lip_worstcase'),
        ('No-MLP + Heuristic Lipschitz',     'dk_lip_heuristic'),
        ('No-MLP + DataDriven Lipschitz',    'dk_lip_datadriven'),
        ('No-MLP + SHAP error-share',        'dk_err_shap'),
        ('No-MLP + ApproxSHAP error-share',  'dk_err_apprxshap'),
        ('kdk (full) + Equal Lipschitz',        'kdk_lip_equal'),
        ('kdk (full) + Random Lipschitz',       'kdk_lip_random'),
        ('kdk (full) + Worst-case Lipschitz',   'kdk_lip_worstcase'),
        ('kdk (full) + Heuristic Lipschitz',    'kdk_lip_heuristic'),
        ('kdk (full) + DataDriven Lipschitz',   'kdk_lip_datadriven'),
        ('kdk (full) + SHAP error-share',       'kdk_err_shap'),
        ('kdk (full) + ApproxSHAP error-share', 'kdk_err_apprxshap'),
    ]
    table = []
    for label, key in order:
        if key in results and len(results[key]['er']) > 0:
            er, vio          = np.array(results[key]['er']), np.array(results[key]['vio']) * 100.0
            umean, lip, smooth = (np.array(results[key][k]) for k in ('umean', 'lip', 'smooth'))
            n_ok = int(np.sum(~np.isnan(er)))
            def fmt(a):
                return f"{np.nanmean(a):.3f} \u00b1 {np.nanstd(a, ddof=1):.3f}" if n_ok > 1 else f"{np.nanmean(a):.3f} \u00b1 n/a"
            table.append([label, fmt(er), fmt(vio), fmt(umean), fmt(lip), fmt(smooth), f"{n_ok}/{N}"])
    print(f"Ablation table (mean +/- std over N={N} trials)")
    print(tabulate(table, headers=["Arm", "MSE loss", "Violation (%)", "Avr. U", "Emp. Lipschitz", "Curvature", "Converged"],
        colalign=("left","center","center","center","center","center","center")))


## Axis 1: MLP block (present vs. absent)

`kdk_full` is the main K2DAREK config. `dk_nomlp` is a standalone `DAREK` with
`width=[1,5,1]` -- the existing "DK2" config, capacity matched to K2DAREK's spline sub-block
in the 1->5->1 sense. Both use `extend=True`/`kan_extend=True` in this notebook variant.

In [ ]:
new_arm('kdk_full'); new_arm('dk_nomlp')
for trial in range(N):
    trial_seed = base_seeds[trial]

    set_seed(trial_seed)
    kdk = K2DAREK(mlp_width=[1, 5], kan_width=[5, 1], kan_grid=8, kan_k=3, kan_base_fun='silu',
                  kan_seed=trial_seed, device='cpu', L_l=np.sqrt(10), symbolic_enabled=False,
                  auto_save=False, kan_extend=True)
    kdk.fit(cos_dataset, lr=0.1, steps=500, lamb=1e-5, nonfixknot=True, seed_knots=trial_seed,
            rand_method='Kmean', scheduler='dec', step_sch=50, gamma=0.9, verbose=False)
    yhat, u = kdk.predict(x_test, L_k=10, L_1=np.sqrt(10), L_mlp=np.sqrt(10))
    record_arm('kdk_full', yhat, u)

    set_seed(trial_seed)
    dk = DAREK(width=[1, 5, 1], grid=8, k=3, base_fun='silu', seed=trial_seed, device='cpu',
               symbolic_enabled=False, auto_save=False, extend=True)
    dk.fit(cos_dataset, lr=0.1, steps=500, lamb=1e-5, nonfixknot=True, seed_knots=trial_seed,
           rand_method='Kmean', scheduler='dec', step_sch=50, gamma=0.9, verbose=False)
    yhat_dk, u_dk = dk.predict(x_test, fk=10, f1=10)
    record_arm('dk_nomlp', yhat_dk, u_dk)

print(f"kdk_full: {len(results['kdk_full']['er'])}/{N} done, dk_nomlp: {len(results['dk_nomlp']['er'])}/{N} done")


## Axis 2: Spectral normalization (on vs. off)

Same config as `kdk_full`, but `use_spectral_norm=False`.

In [ ]:
new_arm('kdk_nospec')
for trial in range(N):
    trial_seed = base_seeds[trial]
    set_seed(trial_seed)
    kdk = K2DAREK(mlp_width=[1, 5], kan_width=[5, 1], kan_grid=8, kan_k=3, kan_base_fun='silu',
                  kan_seed=trial_seed, device='cpu', L_l=np.sqrt(10), symbolic_enabled=False,
                  auto_save=False, kan_extend=True, use_spectral_norm=False)
    kdk.fit(cos_dataset, lr=0.1, steps=500, lamb=1e-5, nonfixknot=True, seed_knots=trial_seed,
            rand_method='Kmean', scheduler='dec', step_sch=50, gamma=0.9, verbose=False)
    yhat, u = kdk.predict(x_test, L_k=10, L_1=np.sqrt(10), L_mlp=np.sqrt(10))
    record_arm('kdk_nospec', yhat, u)

print(f"kdk_nospec: {len(results['kdk_nospec']['er'])}/{N} done")


## Axis 3: Knot-selection strategy

`Kmean` is covered by `kdk_full` above. `freeze_knots(...)` reproduces `fit()`'s own one-time
knot-selection block using an evenly-spaced, non-adaptive `custom` index, then trains with
`nonfixknot=False` so the grid is never re-selected -- see the original ablation notebook's
Axis 3 markdown for why a literal `nonfixknot=False` alone crashes `predict()`.

In [ ]:
def freeze_knots(kdk, dataset, custom_index, seed_knots=0):
    kan = kdk.SNNs
    custom_index = kdk.select_knots(dataset['train_input'], kan.grid, seed=seed_knots,
                                     method='custom', index=custom_index)
    with torch.no_grad():
        y = kdk.forward_mlps(dataset['train_input'])
    kan.forward_update_grid(y, dataset['train_label'], reindex=False, seed=seed_knots,
                             method='custom', index=custom_index)
    if 'xi' not in kan.samples:
        kan.samples['xi'] = dataset['train_input'][kan.samples['indx']]
    kdk.knots   = kan.knots
    kdk.samples = kan.samples

kan_grid = 8
n_train  = x_train.shape[0]
sort_idx = torch.argsort(x_train[:, 0])
fixed_custom_index = sort_idx[np.linspace(0, n_train - 1, kan_grid + 1).round().astype(int)].numpy()

knot_variants = [('kdk_knot_random', 'random', True,  False),
                 ('kdk_knot_lhs',    'LHS',    True,  False),
                 ('kdk_knot_chb',    'chebyshev',    True,  False),
                 ('kdk_knot_wkm',    'gw_kmean',    True,  False),
                 ('kdk_knot_fixed',  None,     False, True)]

for key, rm, nfk, freeze in knot_variants:
    new_arm(key)
    for trial in range(N):
        trial_seed = base_seeds[trial]
        set_seed(trial_seed)
        kdk = K2DAREK(mlp_width=[1, 5], kan_width=[5, 1], kan_grid=kan_grid, kan_k=3, kan_base_fun='silu',
                      kan_seed=trial_seed, device='cpu', L_l=np.sqrt(10), symbolic_enabled=False,
                      auto_save=False, kan_extend=True)
        if freeze:
            freeze_knots(kdk, cos_dataset, fixed_custom_index, seed_knots=trial_seed)
        kdk.fit(cos_dataset, lr=0.1, steps=500, lamb=1e-5, nonfixknot=nfk, seed_knots=trial_seed,
                rand_method=rm, scheduler='dec', step_sch=50, gamma=0.9, verbose=False)
        yhat, u = kdk.predict(x_test, L_k=10, L_1=np.sqrt(10), L_mlp=np.sqrt(10))
        record_arm(key, yhat, u)
    print(f"{key}: {len(results[key]['er'])}/{N} done")


## Axis 4: Spline capacity/order

Same `kdk_full` config, but varying the spline's own flexibility: `k=1` (piecewise-linear),
small (`grid=4`) vs. large (`grid=16`) grid, against the baseline's `grid=8, k=3`.

In [ ]:
capacity_variants = [('kdk_k1', 8, 1), ('kdk_grid_small', 4, 3), ('kdk_grid_large', 14, 3)]

for key, kg, kk in capacity_variants:
    new_arm(key)
    for trial in range(N):
        trial_seed = base_seeds[trial]
        set_seed(trial_seed)
        kdk = K2DAREK(mlp_width=[1, 5], kan_width=[5, 1], kan_grid=kg, kan_k=kk, kan_base_fun='silu',
                      kan_seed=trial_seed, device='cpu', L_l=np.sqrt(10), symbolic_enabled=False,
                      auto_save=False, kan_extend=True)
        kdk.fit(cos_dataset, lr=0.1, steps=500, lamb=1e-5, nonfixknot=True, seed_knots=trial_seed,
                rand_method='Kmean', scheduler='dec', step_sch=50, gamma=0.9, verbose=False)
        yhat, u = kdk.predict(x_test, L_k=10, L_1=np.sqrt(10), L_mlp=np.sqrt(10))
        record_arm(key, yhat, u)
    print(f"{key}: {len(results[key]['er'])}/{N} done")


## Axis 5a: Lipschitz-budget / error-share allocation on the no-MLP spline-only model

Same as the original ablation notebook's Axis 5, plus a new `Random_Lipschitz` baseline
(`Lipschitz_share.py`): randomly splits the Lf1/Lfk budget across layers (Dirichlet-weighted,
rescaled so the aggregate bound matches `Equal_Lipschitz` exactly) -- a naive "non-uniform but
uninformed" contrast against the smarter Heuristic/DataDriven/SHAP allocations.

**`Optimal_Lipschitz` is still excluded**: it reads `self.splines[f'{l}-{i}-{j}']['Ubar_c']`,
but that instrumentation is commented out in `DAREK.predict()`, so every call raises
`KeyError('0-0-0')` -- a pre-existing gap in the shared library, not something introduced here.

In [ ]:
allocation_methods_dk = [
    ('dk_lip_equal',      lambda dk_: Equal_Lipschitz(dk_, x_test, y_test, 10, 10)),
    # ('dk_lip_random',     lambda dk_: Random_Lipschitz(dk_, x_test, y_test, 10, 10)),
    ('dk_lip_worstcase',  lambda dk_: NonOptimal_WorstCase_Lipschitz(dk_, x_test, y_test, 10, 10)),
    ('dk_lip_heuristic',  lambda dk_: Heuristic_Lipschitz(dk_, x_test, y_test, x_train, y_train, 10, 10)),
    ('dk_lip_datadriven', lambda dk_: DataDriven_Lipschitz(dk_, x_test, y_test, x_train, y_train, 10, 10)),
    ('dk_err_shap',       lambda dk_: SHAP_Error_Share(dk_, x_test, y_test, x_train, y_train, 10, 10)),
    ('dk_err_apprxshap',  lambda dk_: Apprx_SHAP_Error_Share(dk_, x_test, y_test, x_train, y_train, 10, 10)),
]
for key, _ in allocation_methods_dk:
    new_arm(key)

for trial in range(N):
    trial_seed = base_seeds[trial]
    set_seed(trial_seed)
    dk = DAREK(width=[1, 5, 1], grid=8, k=3, base_fun='silu', seed=trial_seed, device='cpu',
               symbolic_enabled=False, auto_save=False, extend=True)
    dk.fit(cos_dataset, lr=0.1, steps=500, lamb=1e-5, nonfixknot=True, seed_knots=trial_seed,
           rand_method='Kmean', scheduler='dec', step_sch=50, gamma=0.9, verbose=False)
    for key, fn in allocation_methods_dk:
        set_seed(trial_seed)
        res = fn(dk)
        record_arm(key, res['pred'], res['bound'])

for key, _ in allocation_methods_dk:
    print(f"{key}: {len(results[key]['er'])}/{N} done")


## Axis 5b: Lipschitz-budget / error-share allocation on kdk (full K2DAREK) itself

The paper's claims are about K2DAREK, so this repeats Axis 5a on the actual `kdk` model
instead of only its spline-only ablation. The catch: `Lipschitz_share.py`/`Error_share.py`'s
functions call `self.predict(x, fk, f1, ...)`, matching plain `DAREK.predict()`'s signature --
not `KDAREK.predict(x0, L_mlp, L_k, L_1, ...)`, which has extra params in different
positions. Calling them on `kdk` directly would silently bind `fk`->`L_mlp` and `f1`->`L_k`,
which is wrong, not just incompatible.

`combine_kdk(...)` below instead runs each allocation method on `kdk.SNNs` (the spline block)
using the MLP-transformed representation `y0 = kdk.forward_mlps(x)` as its input, gets back
`(y1, u_spline)`, then re-combines with the MLP block's own error term exactly the way
`KDAREK.predict()` does internally (`opt_K2DAREK.py:115-151`) -- so only the spline block's
budget-allocation strategy varies across arms, while the MLP's own contribution (`L_mlp`,
`L_1`) stays fixed at the `kdk_full` baseline's values, keeping the comparison apples-to-apples.

**Note on interpreting these rows**: K2DAREK's spline sub-block here is `kan_width=[5,1]` --
a single KAN layer (depth=1). `Equal_Lipschitz`/`Heuristic_Lipschitz`/`DataDriven_Lipschitz`/
`Random_Lipschitz` all *redistribute a budget across layers*, so with only one layer to
redistribute across, they collapse to identical numbers here (nothing to allocate). Only
`Worst-case` (which skips the `/NL` budget normalization Equal-family methods use, not just
cross-layer redistribution) and the SHAP-based methods (which allocate across the 5 *edges*
within that single layer) meaningfully differ from Equal for `kdk`. Contrast with Axis 5a
above, where `dk_nomlp` has 2 layers and all six methods differ from each other.

In [ ]:
def mlp_error_term(kdk, x0, L_mlp, L_1_eff, out_dim1):
    xi0 = kdk.SNNs.samples['xi'].unsqueeze(0)
    xt0 = x0.unsqueeze(1)
    min_dist = (xi0 - xt0).abs().min(dim=1)[0]
    mlpw = np.array(kdk.width_mlp)
    L_mlp2 = L_1_eff ** len(mlpw[:-1])
    err_mlp2 = (min_dist * L_mlp2).sum(dim=-1, keepdim=True)
    a = (xi0.max(dim=1)[0] - xi0.min(dim=1)[0]).max() / 6
    err_mlp2 = torch.tanh(err_mlp2 / a) * L_mlp
    return err_mlp2.expand(-1, out_dim1)

def combine_kdk(kdk, x0, y1, u_spline, L_mlp=np.sqrt(10), L_1=np.sqrt(10)):
    depth = len(np.array(kdk.width_mlp)[:-1]) + len(np.array(kdk.width_kan)[:-1])
    L_1_eff = np.power(L_1 / kdk.d, 1 / depth)
    err_mlp2 = mlp_error_term(kdk, x0, L_mlp, L_1_eff, u_spline.shape[1])
    return y1, err_mlp2 * L_1_eff + u_spline

allocation_methods_kdk_keys = ['kdk_lip_equal', 'kdk_lip_random', 'kdk_lip_worstcase',
                               'kdk_lip_heuristic', 'kdk_lip_datadriven',
                               'kdk_err_shap', 'kdk_err_apprxshap']
for key in allocation_methods_kdk_keys:
    new_arm(key)

for trial in range(N):
    trial_seed = base_seeds[trial]
    set_seed(trial_seed)
    kdk = K2DAREK(mlp_width=[1, 5], kan_width=[5, 1], kan_grid=8, kan_k=3, kan_base_fun='silu',
                  kan_seed=trial_seed, device='cpu', L_l=np.sqrt(10), symbolic_enabled=False,
                  auto_save=False, kan_extend=True)
    kdk.fit(cos_dataset, lr=0.1, steps=500, lamb=1e-5, nonfixknot=True, seed_knots=trial_seed,
            rand_method='Kmean', scheduler='dec', step_sch=50, gamma=0.9, verbose=False)

    with torch.no_grad():
        y0_train = kdk.forward_mlps(x_train)
        y0_test  = kdk.forward_mlps(x_test)
    snn = kdk.SNNs

    allocation_methods_kdk = [
        ('kdk_lip_equal',      lambda: Equal_Lipschitz(snn, y0_test, y_test, 10, 10)),
        # ('kdk_lip_random',     lambda: Random_Lipschitz(snn, y0_test, y_test, 10, 10, seed=trial_seed)),
        ('kdk_lip_worstcase',  lambda: NonOptimal_WorstCase_Lipschitz(snn, y0_test, y_test, 10, 10)),
        ('kdk_lip_heuristic',  lambda: Heuristic_Lipschitz(snn, y0_test, y_test, y0_train, y_train, 10, 10)),
        ('kdk_lip_datadriven', lambda: DataDriven_Lipschitz(snn, y0_test, y_test, y0_train, y_train, 10, 10)),
        ('kdk_err_shap',       lambda: SHAP_Error_Share(snn, y0_test, y_test, y0_train, y_train, 10, 10)),
        ('kdk_err_apprxshap',  lambda: Apprx_SHAP_Error_Share(snn, y0_test, y_test, y0_train, y_train, 10, 10)),
    ]
    for key, fn in allocation_methods_kdk:
        set_seed(trial_seed)
        res = fn()
        y1, u = combine_kdk(kdk, x_test, res['pred'], res['bound'], L_mlp=np.sqrt(10), L_1=np.sqrt(10))
        record_arm(key, y1, u)

for key in allocation_methods_kdk_keys:
    print(f"{key}: {len(results[key]['er'])}/{N} done")


## Results

In [ ]:
print_ablation_table()